# Reassign mz indexes to MALDI data

Explanation:
The data in imzml file was processed by R scripts 
of Heiland team (Freiburg):

1. corrected batch effect
2. repeated the rows in the quantifications data to accomodate to the fact that several hmdb correspond to one mz
3. assigned hmdb indexes to data

To remediate 2. and 3. this script re-assigns mz to quantifications data and deduplicates the rows




In [26]:
import os
import numpy as np
import pandas as pd

In [4]:
import zonist.data as zd

INFO:zonist.data:
Time: 20241215-17.59.03


In [5]:
mydata_dir = "/home/johanna/spatial_thesis/spatial_metabolomics/data"
#"/mnt/cbib/thesis_gbm/spatial_thesis/spatial_metabolomics/data"
anno_dict = {'HMDB': "Metaspace_annotations_943peaks.csv"}

In [6]:
me_annots = zd.MetabolomeAnnotations(name="heiland", 
                            data_dir=mydata_dir, 
                            subfolder="heiland-msi",
                            annotation_files=anno_dict)

In [7]:
me_annots.load_annotation()

INFO:zonist.data:Loading annotations
INFO:zonist.data:  loading HMDB


In [8]:
me_annots.annotationtbl.columns

Index(['mz', 'formula', 'moleculeIds', 'moleculeNames', 'database'], dtype='object')

In [18]:
cool = me_annots.annotationtbl[["mz","moleculeIds"]].drop_duplicates()
#cool = cool.set_index("moleculeIds")
hmdb_mz_dict = dict()
for i, r in cool.iterrows():
    hmdb_mz_dict[r["moleculeIds"]] = r["mz"]

In [34]:
samples = [
    "248_T", "259_T", "260_T", "262_T", "269_T", "275_T"
]
#samples = ["248_T"]

In [36]:
for k in samples:
    df = pd.read_csv(
        os.path.join(mydata_dir, "heiland-msi",
                     "ancient",
                 f'MALDI_{k}.tsv'), 
        sep="\t", comment="#", header=0, index_col=0)
    df.insert(0, 'mz', df.index.map(hmdb_mz_dict.get))
    df = df.sort_values(by='mz', ascending=True)
    df.index.name = "hmdb"
    df = df.reset_index()
    df = df.drop(columns=["hmdb"])
    df = df.drop_duplicates()
    df = df.set_index('mz')
    df.to_csv( os.path.join(mydata_dir, "heiland-msi",
                "not_shifted", f'MALDI_{k}.tsv'),  
              sep="\t", header=True, index=True)


## Dealing with negative values resulting from batch effect correction

**Shifting** the distribution of all quantifications, using the abs(min value) of the whole dataframes

In [40]:
print("collecting all min values")
minimals = list()
for k in samples:
    df = pd.read_csv(os.path.join(mydata_dir, "heiland-msi",
                "not_shifted", f'MALDI_{k}.tsv'),  
           sep="\t", comment="#", header=0, index_col=0)
    minimals.append(df.min().min())

collecting all min values


In [41]:
print(minimals)
print(np.array(minimals).min())

[-35708.058040874, -18763.4243632498, -17855.5315492603, -21334.6617616987, -14950.9307982663, -17463.1049660125]
-35708.058040874


In [47]:
if np.array(minimals).min() < 0:
    print("shifting")
    for k in samples:
        df = pd.read_csv(os.path.join(mydata_dir, "heiland-msi",
                 "not_shifted", f'MALDI_{k}.tsv'),
                 sep="\t", header=0, index_col=0)    
        df = df + abs(np.array(minimals).min()) 
        df.to_csv( os.path.join(mydata_dir, "heiland-msi",
             f'MALDI_{k}.tsv'),  
             sep="\t", header=True, index=True)
    

shifting
